In [ ]:
import pandas as pd
and_data = pd.read_csv('ANDHRA_PD.csv')
and_data.head()

In [ ]:
del_data = pd.read_csv('DELHI.csv')
del_data.head()

In [ ]:
kar_data = pd.read_csv('KARNATAKA.csv')
kar_data.head()

In [ ]:
mah_data = pd.read_csv('MAHARASHTRA.csv')
mah_data.head()

In [ ]:
tam_data = pd.read_csv('TAMIL_NADU.csv')
tam_data.head()

In [ ]:
utt_data = pd.read_csv('UTTAR_PD.csv')
utt_data.head()

In [ ]:
def Label(data):
    inc=[]
    means = data['Malaria_Incidence'].mean()
    stds = data['Malaria_Incidence'].std()
    thresh = means + (0.45*stds)
    for i in data['Malaria_Incidence']:
        if i < thresh:
            inc.append(-1)
        else:
            inc.append(1)
    print(inc)
    return inc

In [ ]:
s_data = [del_data,kar_data,mah_data,tam_data,utt_data,and_data]
for i in s_data:
    hh = Label(i)
    i['M_inc'] = hh
    print(i)

In [ ]:
df = and_data.append([del_data,kar_data,mah_data,tam_data,utt_data])
df.head(10)

In [ ]:
from texttable import Texttable
full_headers = df.columns
values = list(df.isnull().sum())
nullList = []
nullList.append(['Feature','Null Values count'])
for i in range(len(full_headers)):
    nullList.append([full_headers[i],values[i]])

table = Texttable()
table.add_rows(nullList)
print(table.draw())    
print("\n")

In [ ]:
import matplotlib.pyplot as plotting
print("The histograms of the attributes are given below:")
df.hist(bins=5,grid=False,layout=[6,6],figsize=[20,20])
plotting.show()
print("\n")

In [ ]:
#k-mean-clustering
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans


x = df.iloc[:, [1, 2, 3, 4, 5, 6, 11]].values

In [ ]:
wcss = []

for i in range(1, 11):
    kmeans = KMeans(n_clusters = i, init = 'k-means++', max_iter = 300, n_init = 10, random_state = 0)
    kmeans.fit(x)
    wcss.append(kmeans.inertia_)
    
#Plotting the results onto a line graph, allowing us to observe 'The elbow'
plt.plot(range(1, 11), wcss)
plt.title('The elbow method')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS') #within cluster sum of squares
plt.show()

In [ ]:
kmeans = KMeans(n_clusters = 2, init = 'k-means++', max_iter = 300, n_init = 10, random_state = 0)
y_kmeans = kmeans.fit_predict(x)


In [ ]:

plt.scatter(x[y_kmeans == 0, 0], x[y_kmeans == 0, 1], s = 100, c = 'blue', label = '1')
plt.scatter(x[y_kmeans == 1, 0], x[y_kmeans == 1, 1], s = 100, c = 'green', label = '-1')

#Plotting the centroids of the clusters
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:,1], s = 100, c = 'yellow', label = 'Centroids')

plt.legend()


In [ ]:
#K-nn
x = df.iloc[:,1:7]
y = df.iloc[:,-1]
#print(x,y)


In [ ]:
import numpy as np
predictors = ['AQI','CO','Temp','Humidity','Precipitation','Population','Rainfall']
crux = "M_inc"

X = df.loc[:,predictors]
Y = np.ravel(df.loc[:,[crux]])

In [ ]:
print(Y)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import  cross_val_score, cross_val_predict

rf = RandomForestClassifier()
sc = cross_val_score(rf, X, Y, scoring='accuracy').mean()
print("Benchmark-> Accuracy before Norm and PCA:- %s"%round(sc*100,2))

In [ ]:
import seaborn as sns
sns.swarmplot(y='Rainfall',x='M_inc', data=df)
plotting.show()
print("\n")

In [ ]:
sns.swarmplot(y='Temp',x='M_inc', data=df)
plotting.show()
print("\n")

In [ ]:
sns.swarmplot(y='AQI',x='M_inc', data=df)
plotting.show()
print("\n")

In [ ]:
import seaborn as sns
b = df
b_corr = b.drop(['Year','Malaria_Cases','Malaria_Death','Dengue_Cases','Kalaazar_Cases','Malaria_Incidence','M_inc'],axis=1)
correlation = b_corr.corr()
heatmap = sns.heatmap(correlation, cbar=True, annot=True, cmap="bwr", linewidths=.75)
heatmap.set_title("Correlation heatmap\n")
plotting.show()

In [ ]:
print(df)

In [ ]:
from sklearn.datasets import make_classification
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier
# define dataset
y = df.drop(columns=['Year','Malaria_Cases','Malaria_Death','Dengue_Cases','Kalaazar_Cases','Malaria_Incidence', 'AQI','CO','Temp','Humidity','Precipitation','Rainfall','Population'],axis=1)
# = x1.values.reshape(1,-1)
x = df.drop(columns=['Year','Malaria_Cases','Malaria_Death','Dengue_Cases','Kalaazar_Cases','Malaria_Incidence','M_inc'],axis=1)
print(x)
print(y)


In [ ]:
# define RFE
rfe = RFE(estimator=DecisionTreeClassifier(), n_features_to_select=3)
# fit RFE
rfe.fit(x, y)
# summarize all features
for i in range(x.shape[1]):
	print('Column: %d, Selected %s, Rank: %.3f' % (i, rfe.support_[i], rfe.ranking_[i]))

#Temp, Humidty and Rainfall takes the top place

In [ ]:
label = df['M_inc']
df = df.drop(columns=['Year','Malaria_Cases','Malaria_Death','Dengue_Cases','Kalaazar_Cases','Malaria_Incidence','M_inc'],axis=1)
headers = df.columns
minimum = list(map(lambda x: round(x,4),df.min()))
mean = list(map(lambda x: round(x,4),df.mean()))
maximum = list(map(lambda x: round(x,4),df.max()))
std =list(map(lambda x: round(x,4),df.std()))

before_scaling=[]
before_scaling.append(['Feature','Min','Mean','Max','Std. Dev'])
before_scaling.append(['M_inci',label.min(),label.mean(),label.max(),label.std()])
for i in range(len(headers)):
    before_scaling.append([headers[i],minimum[i],mean[i],maximum[i],std[i]])

print("\nBEFORE FEATURE SCALING")
table1 = Texttable()
table1.add_rows(before_scaling)
print(table1.draw())
print("\n")

In [ ]:
df.shape

In [ ]:
from sklearn import preprocessing
df = pd.DataFrame(preprocessing.scale(df.iloc[:,0:7]))

minimum = list(map(lambda x: round(x,4),df.min()))
mean = list(map(lambda x: round(x,4),df.mean()))
maximum = list(map(lambda x: round(x,4),df.max()))
std =list(map(lambda x: round(x,4),df.std()))

after_scaling=[]
after_scaling.append(['Feature','Min','Mean','Max','Std. Dev'])
after_scaling.append(['M_inci',label.min(),label.mean(),label.max(),label.std()])
for i in range(len(headers)):
    after_scaling.append([headers[i],minimum[i],mean[i],maximum[i],std[i]])

print("\nAFTER FEATURE SCALING")
table2 = Texttable()
table2.add_rows(after_scaling)
print(table2.draw())
print("\n")

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=len(df.columns))
pca.fit_transform(df)
components = abs(pca.components_)
eigen_values = pca.explained_variance_
ratio_values = pca.explained_variance_ratio_
plotting.ylabel("Eigen values")
plotting.xlabel("Number of features")
plotting.title("PCA eigen values")
plotting.ylim(0, max(eigen_values))
plotting.xticks([1,2,3,4,5,6,7,8,9,10,15,20,25,30])
plotting.style.context('seaborn-whitegrid')
plotting.axhline(y=1,color='r',linestyle='--')
plotting.plot(eigen_values)
plotting.figure(figsize=(500,500))
plotting.show()
print("\n")

In [ ]:
tableList=[]
tableList.append(["NC","SP","EV","CEV"])
for i in range(len(eigen_values)):
    total=0
    for j in range(i+1):
        total+=ratio_values[j]
    tableList.append([i+1,round(eigen_values[i],2),round(ratio_values[i],2),round(total*100,2)])


print("\nPCA Table")
table3 = Texttable()
table3.add_rows(tableList)
print(table3.draw())
print("\n")


In [ ]:
pca_new = PCA(n_components=3)
df = pca_new.fit_transform(df)

X = pd.DataFrame(df)
Y = pd.DataFrame(label)
print(df.shape)
print(X.shape)
print(Y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=0)
print('Original Data Set',df.shape)
print('Shape of X training set : ',X_train.shape,' || Shape of test set : ',X_test.shape) 
print('Shape of Y training set : ',Y_train.shape,' || Shape of test set : ',Y_test.shape) 

In [ ]:
table_report=[]
table_report.append(["Model","Acc","Prec","Recall","F1"])
global count_lis
count_lis = 1

In [ ]:
#PRINT FUNCTION

from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

def truncate(f, n):
    return np.floor(f * 10 ** n) / 10 ** n

def evaluate(sco, model, X_train, X_test, y_train, y_test):
    global count_lis
    y_test_pred = model.predict(X_test)
    y_train_pred = model.predict(X_train)

    clf_report = pd.DataFrame(classification_report(y_train, y_train_pred, output_dict=True))
    acc = round(sco*100,2)
    #print(f"Accuracy %s" % round(accuracy_score(y_train, y_train_pred)*100,2))
    lisp =[]
    lisp = truncate(clf_report.mean(axis = 1).astype(float),2)
    table_report.append([count_lis,acc,lisp['precision'],lisp['recall'],lisp['f1-score']])
    count_lis = count_lis+1

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
#PERCEPTRON
from sklearn.model_selection import  cross_val_score, cross_val_predict
from sklearn.linear_model import Perceptron

pct_clf = Perceptron()
sc = cross_val_score(pct_clf, X_train, Y_train, scoring='accuracy' ,cv=10).mean()

pct_clf.fit(X_train, Y_train)
evaluate(sc, pct_clf, X_train, X_test, Y_train, Y_test)

scores = {
    'Perceptron': {
        'Train': accuracy_score(Y_train, pct_clf.predict(X_train)),
        'Test': accuracy_score(Y_test, pct_clf.predict(X_test)),
    },
}

print("Perceptron Accuracy :- %s" % round(sc*100,2))
print(pct_clf.predict(X_test))

In [ ]:
#XGBOOST

from xgboost import XGBClassifier

xgb_clf = XGBClassifier(eval_metric='mlogloss')
sc = cross_val_score(xgb_clf, X_train, Y_train, scoring='accuracy', cv=10).mean()

xgb_clf.fit(X_train, Y_train)
evaluate(sc, xgb_clf, X_train, X_test, Y_train, Y_test)

scores['xgboost'] = {
        'Train': accuracy_score(Y_train, xgb_clf.predict(X_train)),
        'Test': accuracy_score(Y_test, xgb_clf.predict(X_test)),
    }

print("XGBoost Accuracy :- %s" % round((sc)*100,2))
print(xgb_clf.predict(X_test))

In [ ]:
#KNN
from sklearn.neighbors import KNeighborsClassifier

knn_clf = KNeighborsClassifier()
sc = cross_val_score(knn_clf, X_train, Y_train, scoring='accuracy' ,cv=10).mean()

knn_clf.fit(X_train, Y_train)
evaluate(sc, knn_clf, X_train, X_test, Y_train, Y_test)

scores['KNN'] = {
        'Train': accuracy_score(Y_train, knn_clf.predict(X_train)),
        'Test': accuracy_score(Y_test, knn_clf.predict(X_test)),
    }

print("KNN Accuracy after Norm and PCA :- %s" % round(sc*100,2))
print(knn_clf.predict(X_test))

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
estimators = []

log_flag = LogisticRegression()
estimators.append(('Logistic', log_flag))

pct_flag = Perceptron()
estimators.append(('Percept', pct_flag))

xgb_flag = XGBClassifier(eval_metric='mlogloss')
estimators.append(('XGBboost', xgb_flag))

knn_flag = KNeighborsClassifier()
estimators.append(('KNN', knn_flag))

#By default its hard voting
voting = VotingClassifier(estimators=estimators)
voting.fit(X_train, Y_train)

#acc = round(accuracy_score(Y_train, Y_train_pred)*100,2)
acc = round(cross_val_score(voting, X_train, Y_train ,scoring='accuracy',cv=10).mean()*100,2)

global count_lis
Y_test_pred = voting.predict(X_test)
Y_train_pred = voting.predict(X_train)

clf_report = pd.DataFrame(classification_report(Y_train, Y_train_pred, output_dict=True))

lisp =[]
lisp = truncate(clf_report.mean(axis = 1).astype(float),2)
table_report.append([count_lis,acc,lisp['precision'],lisp['recall'],lisp['f1-score']])

scores['Voting'] = {
        'Train': accuracy_score(Y_train, voting.predict(X_train)),
        'Test': accuracy_score(Y_test, voting.predict(X_test)),
    }


In [ ]:
print("\nPREDICTION RESULTS")
print(" 1. PERCEPTRO\n 2. XGBOOST\n 3. KNN (K-NEAREST NEIGHBOR)\n 4. ENSEMBLE VOTING CLASSIFIER")
table4 = Texttable()
table4.add_rows(table_report)
print(table4.draw())

In [ ]:
scores_df = pd.DataFrame(scores)

scoresList=[]
scoresList.append(["Model","Train Acc","Test Acc","Diff Accuracy"])
for i in scores_df:
    li = list(scores_df[i])
    scoresList.append([i,round(li[0],4),round(li[1],4),"{:.2f}%".format(round(li[1]/li[0],4)*100)])

table5 = Texttable()    
table5.add_rows(scoresList)
print(table5.draw())

In [ ]:
scores_df.plot(kind='bar', figsize=(15, 8))
plotting.show()